In [7]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification,
    pipeline
)
import evaluate
import json
import py_vncorenlp
import os

In [8]:
with open('2.1_train_data.csv', "r", encoding="utf-8") as f:
    df = pd.read_csv(f)
df = df.dropna(subset=['Word', 'Label_BIO', 'Sentence_Id'])
df.head()

,Word,Label_BIO,Sentence_Id
0,HỢP_ĐỒNG,O,0
1,CUNG_CẤP,O,0
2,DỊCH_VỤ,O,0
3,Số,O,1
4,:,O,1


In [9]:
agg_func = lambda s: list(s)
grouped_df = df.groupby('Sentence_Id').agg({
    'Word': agg_func,
    'Label_BIO': agg_func
}).reset_index()
grouped_df.head()

,Sentence_Id,Word,Label_BIO
0,0,"[HỢP_ĐỒNG, CUNG_CẤP, DỊCH_VỤ]","[O, O, O]"
1,1,"[Số, :, 2025, /, HĐDV-023]","[O, O, O, O, O]"
2,3,"[Căn_cứ, Bộ_luật, Dân_sự, năm, 2015, ;]","[O, B-LAW, I-LAW, I-LAW, I-LAW, O]"
3,4,"[Căn_cứ, Luật, Thương_mại, năm, 2005, ;]","[O, B-LAW, I-LAW, I-LAW, I-LAW, O]"
4,6,"[Hôm_nay, ,, ngày, 05, tháng, 12, năm, 2025, ,...","[O, O, B-DATE, I-DATE, I-DATE, I-DATE, I-DATE,..."


In [10]:
sentences = grouped_df['Word'].tolist()
labels = grouped_df['Label_BIO'].tolist()
len(sentences)

475

In [11]:
unique_labels = set(label for doc in labels for label in doc)
label_list = list(unique_labels)
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}
len(unique_labels)

10

In [12]:
model_checkpoint = "vinai/phobert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [13]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = {"input_ids": [], "attention_mask": [], "labels": []}
    max_length = 128

    for i in range(len(examples["tokens"])):
        words = examples["tokens"][i]
        ner_tags = examples["ner_tags"][i]

        # Bắt đầu câu với token <s> (CLS)
        input_ids = [tokenizer.cls_token_id]
        label_ids = [-100]

        for word, label in zip(words, ner_tags):
            # Tokenize từng từ (PhoBERT có thể cắt từ thành các subwords chứa "@@")
            word_tokens = tokenizer.tokenize(word)
            if len(word_tokens) == 0:
                continue
            
            word_input_ids = tokenizer.convert_tokens_to_ids(word_tokens)
            input_ids.extend(word_input_ids)
            
            # Gán nhãn cho subword ĐẦU TIÊN, các subword sau gán -100 để ignore
            label_ids.append(label2id[label])
            label_ids.extend([-100] * (len(word_input_ids) - 1))

        # Kết thúc câu với token </s> (SEP)
        input_ids.append(tokenizer.sep_token_id)
        label_ids.append(-100)
        
        # Cắt ngắn (Truncation) nếu câu quá dài, hoặc thêm Padding nếu câu ngắn
        if len(input_ids) > max_length:
            # Giữ lại token SEP ở cuối cùng nếu bị cắt
            input_ids = input_ids[:max_length-1] + [tokenizer.sep_token_id]
            label_ids = label_ids[:max_length-1] + [-100]
            attention_mask = [1] * max_length
        else:
            pad_len = max_length - len(input_ids)
            attention_mask = [1] * len(input_ids) + [0] * pad_len
            input_ids = input_ids + [tokenizer.pad_token_id] * pad_len
            label_ids = label_ids + [-100] * pad_len

        tokenized_inputs["input_ids"].append(input_ids)
        tokenized_inputs["attention_mask"].append(attention_mask)
        tokenized_inputs["labels"].append(label_ids)

    return tokenized_inputs

In [14]:
hf_dataset = Dataset.from_dict({"tokens": sentences, "ner_tags": labels})
hf_dataset = hf_dataset.train_test_split(test_size=0.1)
tokenized_datasets = hf_dataset.map(tokenize_and_align_labels, batched=True)

Map: 100%|██████████| 48/48 [00:00<00:00, 3185.55 examples/s]


In [15]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, 
    num_labels=len(label_list), 
    id2label=id2label, 
    label2id=label2id
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval = evaluate.load("seqeval")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 39517.81it/s]
RobertaForTokenClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    if not results:
        raise Exception("No results")
    return {
        "precision": results["overall_precision"], 
        "recall": results["overall_recall"], 
        "f1": results["overall_f1"], 
        "accuracy": results["overall_accuracy"]
    }

In [17]:
training_args = TrainingArguments(
    output_dir=f"./phobert-ner-contract",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10, 
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10
)
trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer, 
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model("./phobert-ner-contract-final")

c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.720559,0.634051,0.000000,0.000000,0.000000,0.860544
2,0.474940,0.430641,0.652174,0.428571,0.517241,0.903061
3,0.295167,0.310829,0.818182,0.514286,0.631579,0.945578
4,0.206412,0.278806,0.657143,0.657143,0.657143,0.940476
5,0.126808,0.246175,0.743590,0.828571,0.783784,0.954082
6,0.139184,0.220467,0.833333,0.857143,0.845070,0.967687
7,0.136206,0.212636,0.837838,0.885714,0.861111,0.967687
8,0.105722,0.221558,0.815789,0.885714,0.849315,0.965986
9,0.095764,0.190453,0.885714,0.885714,0.885714,0.971088
10,0.098604,0.191989,0.861111,0.885714,0.873239,0.971088


c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.10it/s]
c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\thien\OneDrive\Desktop\Ass_NLP\.venv\Lib\s

# Predict

In [18]:
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-26" 
try:
    path # type: ignore
except:
    path=os.getcwd()
    ViTokenizer = py_vncorenlp.VnCoreNLP(save_dir=f"{path}\\VnCoreNLP", annotators=["wseg", "pos", "parse"]) # type: ignore
    os.chdir("..")

In [19]:
model_path = "./phobert-ner-contract-final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
ner_pipeline = pipeline(task="ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple") # type: ignore

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9560.79it/s]


In [20]:
def predict_entities(text):
    # 1. Xử lý Word Segmentation bằng VnCoreNLP
    # VnCoreNLP trả về một list các câu (ví dụ: ['Bên_A phải thanh_toán', '...'])
    segmented_sentences = ViTokenizer.word_segment(text)
    
    # 2. Ghép các câu lại thành một chuỗi duy nhất cách nhau bởi khoảng trắng
    segmented_text = " ".join(segmented_sentences)
    
    # 3. Đưa vào mô hình dự đoán
    raw_results = ner_pipeline(segmented_text)
    
    entities = []
    
    # Do aggregation_strategy="simple", kết quả trả về là một list các từ điển (dicts)
    cur_word=""
    cur_label=""
    for res in raw_results:
        label = res['entity_group']
        
        # Bỏ qua nhãn O (Outside)
        if label == "O":
            continue
            
        # Làm sạch kết quả: 
        # - Xóa dấu gạch dưới do VnCoreNLP tạo ra
        # - Xóa các ký hiệu subword @@ đặc trưng của tokenizer PhoBERT
        word = res['word'].replace("_", " ")
        if "@@" in word or "@@ " in word:
            word=word.replace("@@", "").replace("@@ ", "")
            cur_word+=word
            cur_label=cur_label if cur_label else label
            continue
        cur_word+=word
        entities.append({
            "text": cur_word,
            "label": cur_label if cur_label else label
        })
        cur_word=""
        cur_label=""
        
    return entities

In [21]:
raw_texts= []
with open(f"output/clauses.txt", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip() == "":
            continue
        raw_texts.append(line.strip())
raw_texts

['Bên A cung cấp dịch vụ phát triển Hệ thống Quản lý Bệnh viện Điện tử ( HIS ) phiên bản 3.0 .',
 'Phạm vi : phân tích , thiết kế , phát triển , triển khai , đào tạo và bảo trì 24 tháng .',
 'Bên A cam kết hệ thống đáp ứng tiêu chuẩn bảo mật dữ liệu y tế theo Bộ Y tế .',
 'Bên A báo cáo tiến độ mỗi 02 tuần và họp review hàng tháng .',
 'Chậm tiến độ quá 15 ngày , Bên B có quyền yêu cầu tăng nhân lực hoặc phạt theo Điều 7 .',
 'Bên A không được sử dụng mã nguồn tuỳ chỉnh cho bên thứ ba .',
 'Bên B có quyền yêu cầu chuyển giao toàn bộ tài liệu kỹ thuật .',
 'Bên A cam kết bảo mật dữ liệu bệnh nhân , hồ sơ y tế và thông tin kinh doanh .',
 'Nghiêm cấm Bên A truy cập , sao chép hoặc chia sẻ dữ liệu bệnh nhân cho bên thứ ba .',
 'Nghĩa vụ bảo mật kéo dài vô thời hạn đối với dữ liệu y tế .',
 'Vi phạm bảo mật : bồi thường tối thiểu 500.000.000 VNĐ .',
 'Bảo hành 24 tháng kể từ ngày nghiệm thu .',
 'Hỗ trợ kỹ thuật 24/7 trong thời gian bảo hành .',
 'Chậm tiến độ : phạt 0.3% giá trị hợp đồng 

In [23]:
final_output = []

print("\n--- ĐANG TIẾN HÀNH NHẬN DẠNG THỰC THỂ ---")

for clause in raw_texts:
    # Dự đoán thực thể cho từng mệnh đề
    extracted_entities = predict_entities(clause)
    
    # Tạo cấu trúc JSON cho mỗi câu theo đúng mẫu yêu cầu
    # Lưu ý: Mỗi entry là một dict chứa danh sách "entities"
    sentence_entry = {
        "text": clause, # Bao gồm câu gốc để dễ đối chiếu
        "entities": extracted_entities
    }
    final_output.append(sentence_entry)

# Tạo thư mục output nếu chưa có
os.makedirs("output", exist_ok=True)

# Ghi dữ liệu vào file JSON
output_path = "output/ner_results.json"
with open(output_path, "w", encoding="utf-8") as f:
    # ensure_ascii=False để hiển thị được tiếng Việt
    # indent=4 để file JSON dễ đọc
    json.dump(final_output, f, ensure_ascii=False, indent=4)

print(f"\n✅ Đã tạo file thành công tại: {output_path}")
print(f"Tổng số mệnh đề đã xử lý: {len(final_output)}")


--- ĐANG TIẾN HÀNH NHẬN DẠNG THỰC THỂ ---

✅ Đã tạo file thành công tại: output/ner_results.json
Tổng số mệnh đề đã xử lý: 166
